In [15]:
uncond_128 = {
    "keydiff": {
        "mmd": 0.0010251998901367188,
        "std_mmd": 0.00014177958170572916,
        "W_2": 0.19614737729231516,
        "std_w_2": 0.008563233746422658,
    },
    "regression": {
        "mmd": 8.245309193929036e-05,
        "std_mmd": 5.393558078342015e-05,
        "W_2": 0.037655360996723175,
        "std_w_2": 0.005762229363123576,
    },
    "fsbm": {
        "mmd": 0.00015878677368164062,
        "std_mmd": 7.891654968261719e-05,
        "W_2": 0.00933530724917849,
        "std_w_2": 0.0019389989061488044,
    },
    "cnfss": {
        "mmd": 0.0007073879241943359,
        "std_mmd": 8.702278137207031e-05,
        "W_2": 0.038431608428557716,
        "std_w_2": 0.001038378311528099,
    },
    "cnf": {
        "mmd": 0.00039764245351155597,
        "std_mmd": 0.00017113155788845485,
        "W_2": 0.00024167199929555258,
        "std_w_2": 0.029998663398954604,
    },
    "cgan": {
        "mmd": 0.006666819254557292,
        "std_mmd": 0.0005568928188747829,
        "W_2": 0.21776433289051056,
        "std_w_2": 0.01310431957244873,
    },
    "ugan": {
        "mmd": 0.0004660288492838542,
        "std_mmd": 0.00018813874986436629,
        "W_2": 0.019341467569271725,
        "std_w_2": 0.004430095768637127,
    },
    "gcot": {
        "mmd": -2.8371810913085938e-05,
        "std_mmd": 1.7642974853515625e-05,
        "W_2": -0.005698660388588905,
        "std_w_2": 0.009942855685949326,
    },
}

In [18]:
# value = mantissa × 10^exp  (mantissa rounded to `decimals` digits after '.')
# Each key applies to the metric and its std column.
BASELINE_SCALES = {
    "mmd": -3,  # MMD, ± std MMD
    "W_2": -1,  # W₂, ± std W₂
}
BASELINE_DECIMALS = 2

_METRIC_PAIRS = (("mmd", "std_mmd"), ("W_2", "std_w_2"))

def _fmt_scaled(x: float, exp: int, *, decimals: int) -> str:
    quantum = 10.0 ** (exp - decimals)
    x = round(x / quantum) * quantum
    mantissa = x / (10.0**exp)
    return f"{mantissa:+.{decimals}f}e{exp:+03d}"

def print_baseline_table(
    data: dict,
    *,
    scales: dict[str, int] | None = None,
    decimals: int = BASELINE_DECIMALS,
    title: str = "Unconditional (N=128)",
) -> None:
    scales = BASELINE_SCALES if scales is None else scales
    missing = [k for k, _ in _METRIC_PAIRS if k not in scales]
    if missing:
        raise KeyError(f"Missing exponent(s) in scales: {missing}")

    cols = ("Method", "MMD", "± std MMD", "W₂", "± std W₂")
    fmt = {
        key: [_fmt_scaled(m[key], scales[value_key], decimals=decimals) for m in data.values()]
        for value_key, std_key in _METRIC_PAIRS
        for key in (value_key, std_key)
    }
    metric_keys = [k for pair in _METRIC_PAIRS for k in pair]
    col_w = [12] + [max(len(s) for s in fmt[key]) for key in metric_keys]
    sep, rule = "─", "═"

    def row(cells: tuple[str, ...]) -> str:
        return "  " + "  ".join(
            cells[i].ljust(col_w[i]) if i == 0 else cells[i].rjust(col_w[i])
            for i in range(len(cells))
        )

    width = 2 + sum(col_w) + 2 * (len(col_w) - 1)
    print()
    print(rule * width)
    print(f"  {title}".center(width))
    print(rule * width)
    print(row(cols))
    print("  " + sep * col_w[0] + "  " + "  ".join(sep * col_w[i] for i in range(1, len(col_w))))
    for i, (method, _) in enumerate(data.items()):
        print(
            row(
                (
                    method,
                    fmt["mmd"][i],
                    fmt["std_mmd"][i],
                    fmt["W_2"][i],
                    fmt["std_w_2"][i],
                )
            )
        )
    print(rule * width)
    print()

print_baseline_table(uncond_128)



══════════════════════════════════════════════════════════
                   Unconditional (N=128)                  
══════════════════════════════════════════════════════════
  Method              MMD  ± std MMD         W₂   ± std W₂
  ────────────  ─────────  ─────────  ─────────  ─────────
  keydiff       +1.03e-03  +0.14e-03  +1.96e-01  +0.09e-01
  regression    +0.08e-03  +0.05e-03  +0.38e-01  +0.06e-01
  fsbm          +0.16e-03  +0.08e-03  +0.09e-01  +0.02e-01
  cnfss         +0.71e-03  +0.09e-03  +0.38e-01  +0.01e-01
  cnf           +0.40e-03  +0.17e-03  +0.00e-01  +0.30e-01
  cgan          +6.67e-03  +0.56e-03  +2.18e-01  +0.13e-01
  ugan          +0.47e-03  +0.19e-03  +0.19e-01  +0.04e-01
  gcot          -0.03e-03  +0.02e-03  -0.06e-01  +0.10e-01
══════════════════════════════════════════════════════════



In [ ]:
%load_ext autoreload
%autoreload 2


## 1. Imports


In [ ]:
from src.utils.notebook_setup import ensure_repo_imports, load_train_builders

REPO_ROOT = ensure_repo_imports()
build_gmm_model, build_neural_model = load_train_builders(REPO_ROOT)
import math
import random
import sys
from pathlib import Path

import torch
from comet_ml import Experiment
from omegaconf import OmegaConf
from tqdm import tqdm

from src.utils.training import (
    CometExperiment,
    build_ema_model_copy,
    build_periodic_loss_payload,
    build_run_metadata,
    build_swiss_roll_context,
    compose_swiss_roll_cfg,
    compute_metrics,
    log_optional_metrics,
    make_adam,
    should_run,
    update_average,
)
from src.utils.evaluation.metrics import (
    compute_mmd,
    compute_sinkhorn_divergence,
    median_heuristic,
    mixture_kernel,
    rbf_kernel,
)
from src.utils.datasets.match import get_GT_points, load_or_compute_gt_points
from src.utils.plotting.distributions import plot_swiss_roll
from src.utils.plotting.parameters import (
    plot_A_parameters,
    plot_B_parameters,
    plot_Z_parameters,
)


In [ ]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device


In [ ]:
torch.set_default_device(device)
dtype = torch.float64
torch.set_default_dtype(dtype)


## 2. Config


In [ ]:
# Papermill: EXPERIMENT selects conf/experiment/<name>.yaml; OVERRIDES are Hydra CLI-style strings.
# Keep this tagged parameters cell limited to parameter declarations only.
# Papermill injects values in a new cell *after* this one.
EXPERIMENT = "gmm-swiss-roll"  # gmm-swiss-roll | gmm-swiss-roll-shared
OVERRIDES = [
    # "train.steps_to=3000",
    # "ebieot.model.n_potentials=50",
]

def _extract_override_int(overrides: list[str], key: str) -> int | None:
    prefix = f"{key}="
    for item in overrides:
        if item.startswith(prefix):
            return int(item[len(prefix) :])
    return None


In [ ]:
# Compose Hydra config after Papermill injected-parameters cell has executed.
# LSE cost uses no hidden MLP layers by default (empty hidden_channels in
# conf/experiment/gmm_swiss_roll.yaml); only n_potentials / m_potentials vary in sweeps.
if isinstance(OVERRIDES, str):
    OVERRIDES = [OVERRIDES]
else:
    OVERRIDES = [str(item) for item in OVERRIDES]

EXPERIMENT_ALIASES = {
    "gmm-swiss-roll": "gmm_swiss_roll",
    "gmm-swiss-roll-shared": "gmm_swiss_roll_shared",
}
cfg, EXPERIMENT_KEY, seed = compose_swiss_roll_cfg(
    str(REPO_ROOT),
    str(EXPERIMENT),
    OVERRIDES,
    aliases=EXPERIMENT_ALIASES,
)

requested_n = _extract_override_int(OVERRIDES, "ebieot.model.n_potentials")
requested_m = _extract_override_int(OVERRIDES, "ebieot.cost.m_potentials")
effective_n = int(cfg.ebieot.model.n_potentials)
effective_m = int(cfg.ebieot.cost.m_potentials)
override_match = (
    (requested_n is None or effective_n == requested_n)
    and (requested_m is None or effective_m == requested_m)
)
if requested_n is not None and effective_n != requested_n:
    raise RuntimeError(
        f"n_potentials override was not applied: requested={requested_n}, "
        f"effective={effective_n}"
    )
if requested_m is not None and effective_m != requested_m:
    raise RuntimeError(
        f"m_potentials override was not applied: requested={requested_m}, "
        f"effective={effective_m}"
    )

paired_cfg = cfg.train.optimizer.paired
unpaired_cfg = cfg.train.optimizer.unpaired
RUN_METADATA = build_run_metadata("EBiEOT-SwissRoll-GMM", EXPERIMENT_KEY, cfg)

print(f"EXPERIMENT input: {EXPERIMENT}")
print(f"OVERRIDES input: {OVERRIDES}")
print(f"Experiment key: {RUN_METADATA['experiment_key']}")
print(f"Cost preset: {RUN_METADATA['cost_function_label']}")
print(f"Seed: {seed}")
print(f"requested n_potentials: {requested_n}, effective n_potentials: {effective_n}")
print(f"requested m_potentials: {requested_m}, effective m_potentials: {effective_m}")
print(f"override_match: {override_match}")
print(OmegaConf.to_yaml(cfg.train))

In [ ]:
COST_FUNCTION = RUN_METADATA["cost_function_label"]
SHARED_PRESET = RUN_METADATA["shared_preset"]


Config is composed from `conf/experiment/` via Hydra `compose` (see §2).

| `EXPERIMENT` | Role |
| --- | --- |
| `gmm-swiss-roll` | MLPLSE cost, 128 paired / 1k unpaired (legacy `ebieot_gmm_swiss_roll`) |
| `gmm-swiss-roll-shared` | SharedMLPLSE, 16k marginals (legacy `ebieot_gmm_swiss_roll_another_plan`) |


## 3. Model & data


In [ ]:
model = build_gmm_model(cfg, device)
context = build_swiss_roll_context(cfg, device)

usd_sampler = context["usd_sampler"]
utd_sampler = context["utd_sampler"]
pd_sampler = context["pd_sampler"]
x_sampler = context["x_sampler"]
y_sampler = context["y_sampler"]
otp_sampler = context["otp_sampler"]
X_paired_train = context["X_paired_train"]
Y_paired_train = context["Y_paired_train"]
X_paired_test = context["X_paired_test"]
Y_paired_test = context["Y_paired_test"]
X_unpaired_test = context["X_unpaired_test"]
Y_unpaired_test = context["Y_unpaired_test"]

ds = cfg.dataset
model.init_a_by_samples(y_sampler.sample(int(cfg.ebieot.model.n_potentials)))
model_copy = build_ema_model_copy(build_gmm_model, cfg, device, model, bool(cfg.train.ema_update))


## 4. Optimizers


In [ ]:
if SHARED_PRESET:
    unpaired_params = [
        {"params": [model._log_w_n, model._a_n], "lr": float(unpaired_cfg.lr)},
        {"params": [model._log_A_n], "lr": float(unpaired_cfg.lr) * 0.1},
    ]
else:
    unpaired_params = [model._log_w_n, model._a_n, model._log_A_n]

D_opt_unpaired = make_adam(unpaired_params, unpaired_cfg)
D_opt_paired = make_adam(model.cost.parameters(), paired_cfg)


In [ ]:
train_cfg = cfg.train
ds = cfg.dataset
effective_n_potentials = int(cfg.ebieot.model.n_potentials)
effective_m_potentials = int(cfg.ebieot.cost.m_potentials)
EXP_NAME = f"{RUN_METADATA['run_name']}-n{effective_n_potentials}-m{effective_m_potentials}"
OUTPUT_PATH = REPO_ROOT / "checkpoints" / EXP_NAME
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

experiment = Experiment(project_name="ebieot")
experiment.set_name(EXP_NAME)
experiment.log_parameters(OmegaConf.to_container(cfg, resolve=False))
experiment.log_parameters(
    {
        "sensitivity.requested_n_potentials": requested_n,
        "sensitivity.requested_m_potentials": requested_m,
        "sensitivity.effective_n_potentials": effective_n_potentials,
        "sensitivity.effective_m_potentials": effective_m_potentials,
        "sensitivity.override_match": bool(override_match),
    }
)
comet_experiment = CometExperiment(experiment)

if int(train_cfg.steps_from) > 0:
    D_opt_unpaired.load_state_dict(
        torch.load(OUTPUT_PATH / f"D_opt_unpaired_{train_cfg.steps_from}.pt", map_location=device)
    )
    D_opt_paired.load_state_dict(
        torch.load(OUTPUT_PATH / f"D_opt_paired_{train_cfg.steps_from}.pt", map_location=device)
    )


### Training loop (EBiEOT-GMM)

Each step minimizes the entropic OT dual objective. **Paired** batches fit supervised couplings; **unpaired** batches use marginal samples and the learned GMM dual \(f\) and cost. The discrete OT plan sampler provides \(X\) indices for unpaired \(Y\) updates when configured.

## 5. Training


In [ ]:
starting_points = torch.tensor([[-2.0, 0.0], [2.0, 2.0], [0.0, 0.0]], device=device)
num_ending_points = 64

num_starting_points_paired = 5
indices = random.choices(range(X_paired_train.shape[0]), k=num_starting_points_paired)
starting_points_paired = X_paired_train[indices]
ending_points_paired = Y_paired_train[indices]

gt_Y_points = get_GT_points(x_sampler, y_sampler, otp_sampler, starting_points, num_ending_points)
gt_Y_points_for_metrics = load_or_compute_gt_points(
    starting_points,
    x_sampler,
    y_sampler,
    otp_sampler,
    compute_func=get_GT_points,
    num_ending_points=1024,
)

base = median_heuristic(X_paired_test, Y_paired_test)
print(f"Base for MMD metric: {base:.3f}")

kernel_mul = 2.0
kernel_num = 5
base /= kernel_mul ** (kernel_num // 2)
bandwidth_list = [base.item() * (kernel_mul**i) for i in range(kernel_num)]
print(f"Bandwidth list: {bandwidth_list}")

kernel = lambda x, y: mixture_kernel(
    x,
    y,
    [lambda x, y, sigma=sigma: rbf_kernel(x, y, sigma=sigma) for sigma in bandwidth_list],
)

metrics_dict = {
    "mmd": lambda x, y: compute_mmd(x, y, kernel=kernel),
    "sinkhorn": lambda x, y: compute_sinkhorn_divergence(x, y),
}

max_norm = float(train_cfg.gradient_max_norm)
clip_grads = math.isfinite(max_norm)
num_metric_samples = 1024

In [ ]:
final_unconditional_metrics = None

for step in tqdm(range(int(train_cfg.steps_from), int(train_cfg.steps_to))):
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(int(train_cfg.unpaired_batch_size))
    Y = utd_sampler.sample(int(train_cfg.unpaired_batch_size))
    output_unpaired = model.compute_unpaired_loss(X, Y)
    D_loss_unpaired = output_unpaired["loss"]

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_sampler.sample(int(train_cfg.paired_batch_size))
    output_paired = model.compute_paired_loss(X_paired, Y_paired)
    D_loss_paired = output_paired["loss"]

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()

    if clip_grads:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
        torch.nn.utils.clip_grad_norm_(model.cost.parameters(), max_norm=max_norm)

    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_cfg.ema_update and model_copy is not None:
        update_average(model_copy, model, 0.99)
        model = model_copy

    log_every = int(train_cfg.get("log_every", 0))
    if should_run(step, log_every):
        periodic_loss_payload = build_periodic_loss_payload(
            model,
            output_unpaired,
            output_paired,
            D_loss,
            X_paired_train,
            Y_paired_train,
            X_paired_test,
            Y_paired_test,
            X_unpaired_test,
            Y_unpaired_test,
        )
        experiment.log_metrics(periodic_loss_payload, step=step)
        log_optional_metrics(
            experiment,
            output_unpaired,
            {
                "f_c": ("-f^c(x)", lambda t: -t.mean().item()),
                "f": ("-f(y)", lambda t: -t.mean().item()),
                "A_n": ("lam_min(A_n)", lambda t: float(torch.min(t))),
            },
            step,
        )
        if "A_n" in output_unpaired:
            experiment.log_metric("lam_max(A_n)", float(torch.max(output_unpaired["A_n"])), step=step)
        final_unconditional_metrics, _ = compute_metrics(
            models_dict={"ebieot_gmm": model},
            metrics_dict=metrics_dict,
            X_sampler=x_sampler,
            Y_sampler=y_sampler,
            starting_points=starting_points,
            gt_Y_points=gt_Y_points_for_metrics,
            num_samples=num_metric_samples,
            experiment=comet_experiment,
        )

    plot_every = int(train_cfg.get("plot_every", 0))
    if should_run(step, plot_every):
        plot_A_parameters(model, experiment=comet_experiment)
        plot_B_parameters(model.cost, starting_points, experiment=comet_experiment)
        if num_starting_points_paired > 0:
            plot_Z_parameters(
                model,
                starting_points,
                starting_points_paired,
                ending_points_paired,
                experiment=comet_experiment,
            )
        else:
            plot_Z_parameters(model, starting_points, experiment=comet_experiment)
        plot_swiss_roll(
            {f"P={ds.P_XY_paired}, Q={ds.Q_X_unpaired}, R={ds.R_Y_unpaired}": model},
            x_sampler,
            y_sampler,
            X_paired,
            Y_paired,
            starting_points,
            gt_Y_points,
            experiment=comet_experiment,
        )
        torch.save(model.state_dict(), OUTPUT_PATH / f"model_{step}.pt")

torch.save(model.state_dict(), OUTPUT_PATH / f"D_{train_cfg.steps_to}.pt")
torch.save(D_opt_paired.state_dict(), OUTPUT_PATH / f"D_opt_paired_{train_cfg.steps_to}.pt")
torch.save(D_opt_unpaired.state_dict(), OUTPUT_PATH / f"D_opt_unpaired_{train_cfg.steps_to}.pt")

experiment.end()


## 6. Optuna metric (scrapbook)


In [ ]:
import scrapbook as sb

if final_unconditional_metrics is None:
    final_unconditional_metrics, final_conditional_metrics = compute_metrics(
        models_dict={"ebieot_gmm": model},
        metrics_dict=metrics_dict,
        X_sampler=x_sampler,
        Y_sampler=y_sampler,
        starting_points=starting_points,
        gt_Y_points=gt_Y_points_for_metrics,
        num_samples=num_metric_samples,
    )
else:
    final_unconditional_metrics, final_conditional_metrics = compute_metrics(
        models_dict={"ebieot_gmm": model},
        metrics_dict=metrics_dict,
        X_sampler=x_sampler,
        Y_sampler=y_sampler,
        starting_points=starting_points,
        gt_Y_points=gt_Y_points_for_metrics,
        num_samples=num_metric_samples,
    )

mmd_metric = float(final_unconditional_metrics["ebieot_gmm"]["mmd"])
sinkhorn_metric = float(final_unconditional_metrics["ebieot_gmm"]["sinkhorn"])
conditional_mmd_metric = float(final_conditional_metrics["ebieot_gmm"]["mmd"])
conditional_sinkhorn_metric = float(final_conditional_metrics["ebieot_gmm"]["sinkhorn"])
target_metric = mmd_metric
effective_n_potentials = int(cfg.ebieot.model.n_potentials)
effective_m_potentials = int(cfg.ebieot.cost.m_potentials)
sb.glue("target_metric", target_metric)
sb.glue("mmd_metric", mmd_metric)
sb.glue("sinkhorn_metric", sinkhorn_metric)
sb.glue("conditional_mmd_metric", conditional_mmd_metric)
sb.glue("conditional_sinkhorn_metric", conditional_sinkhorn_metric)
sb.glue("effective_n_potentials", effective_n_potentials)
sb.glue("effective_m_potentials", effective_m_potentials)
{
    "unconditional": {"mmd": mmd_metric, "sinkhorn": sinkhorn_metric},
    "conditional": {
        "mmd": conditional_mmd_metric,
        "sinkhorn": conditional_sinkhorn_metric,
    },
}


## 7. Plotting


In [ ]:
plot_swiss_roll(
    {"ebieot_gmm": model},
    x_sampler,
    y_sampler,
    X_paired_train,
    Y_paired_train,
    starting_points,
    gt_Y_points,
    num_ending_points=num_ending_points,
)
